In [ ]:
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')

In [ ]:
symlist = self.get_updated_symbol_list(age_ub=60)
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
df_earning = self.count_days_from_earning_reports(df_quotes)
if df_earning.shape[0] == 0:
    d2e = {}
else:
    d2e = df_earning['earningDays'].to_dict()
print('Days to E:', d2e)
df_raw = self.build_option_df(symlist)
px.bar(self.check_data_age(df_raw), barmode='group', width=60*len(symlist), height=300).show()

In [ ]:
df_put = self.select_options_by_type(df_raw, 'put')
self.plot_option_stats(df_put)
pp = ParallelOptionCalculator(df_put, self, f'/run/user/{os.getuid()}/time_decay_put')
csv_files = pp.do_all_theta_curves(symlist)
dfp = pp.assemble_time_decay_df(csv_files)

### Put options with no earning date on or before expiration date

In [ ]:
hdte_resid_ub = 0.5
spread_ub = 10
moneyness_ub = 0.99
premium_lb = 2
delta_lb = -0.3
_filter = (dfp.moneyness <= moneyness_ub) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub)
_filter = _filter & (dfp.mid >= premium_lb) & (dfp.Delta >= delta_lb)
_filter = _filter & (dfp.E.isna() |(dfp.E > dfp.dte)) # Note: Fidelity's earning report dates are not reliable
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
#_filter = _filter & (dfp.hdteProfit >= 20) & (dfp.strike <= 150)
#_dfp = dfp[_filter].sort_values(by='dth')
print('Options after the filters:', _dfp.shape[0])
px.scatter(_dfp.head(120), x='Delta', y='hdteProfit', color='symbol', height=500).show()
_dfp.head(60)

### Put options for specific symbols

In [ ]:
_filter = dfp.symbol.str.contains('QQQ') & (dfp.moneyness >= 0.9) & (dfp.moneyness <= 1) & (dfp.Delta >= -0.3)
_filter = _filter & (dfp.mid >= 2)
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
px.scatter(_dfp, x='Delta', y='hdteProfit', color='expDt', height=500).show()
print(_dfp.shape)
_dfp.head(25)

### Put options: top 500 in-the-money

In [ ]:
px.scatter(dfp[(dfp.moneyness <= 1) & (dfp.hdteProfit <= 100)].sort_values(by='hdteProfit', ascending=False).head(20), x='hdte_resid', y='hdteProfit', color='symbol', height=600)